# RecoMart Data Ingestion Pipeline

## Overview
This notebook implements the data ingestion layer for the RecoMart recommendation system pipeline. It handles automated ingestion from multiple data sources with comprehensive error handling, retry logic, and audit logging.

## Data Sources
1. **User Interactions (CSV)**: User clickstream and interaction data including views, clicks, and engagement metrics
2. **Transactions (CSV)**: Purchase history and transaction records with order details
3. **Products (JSON)**: Product catalog metadata including categories, descriptions, and attributes

## Pipeline Features
- ✅ Automated retry logic for transient failures
- ✅ Comprehensive logging and audit trails
- ✅ Delta Lake storage with time-travel capabilities
- ✅ Partitioned storage for efficient querying
- ✅ Ingestion metadata tracking (timestamps, source files)
- ✅ Data validation and error reporting

## Target Storage
All data is ingested into Unity Catalog Delta tables:
- Catalog: `recomart`
- Schema: `raw`
- Tables: `user_interactions`, `transactions`, `products`

In [0]:
# Import required libraries
import pandas as pd
import json
import logging
import os
import time
from datetime import datetime, timedelta
from functools import wraps
from typing import Optional, Dict, Any
import traceback

# Configure logging
log_dir = "/Workspace/Users/2025ae05415@wilp.bits-pilani.ac.in/RecoMart_Recommendation_Pipeline/logs"
os.makedirs(log_dir, exist_ok=True)

log_file = f"{log_dir}/ingestion.log"

# Set up logging configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()  # Also log to console
    ]
)

logger = logging.getLogger('RecoMartIngestion')

# Define project paths
PROJECT_ROOT = "/Workspace/Users/2025ae05415@wilp.bits-pilani.ac.in/RecoMart_Recommendation_Pipeline"
DATA_ROOT = f"{PROJECT_ROOT}/data"
RAW_DATA_DIR = f"{DATA_ROOT}/raw"

logger.info("="*80)
logger.info("RecoMart Data Ingestion Pipeline - Session Started")
logger.info(f"Timestamp: {datetime.now().isoformat()}")
logger.info("="*80)

print("✅ Setup complete - Logging initialized")
print(f"📁 Log file: {log_file}")
print(f"📁 Data root: {DATA_ROOT}")

In [0]:
# Data source paths
DATA_SOURCES = {
    'user_interactions': {
        'path': f"{RAW_DATA_DIR}/user_interactions",
        'type': 'csv',
        'target_table': 'recomart.raw.user_interactions',
        'partition_by': 'interaction_date'
    },
    'transactions': {
        'path': f"{RAW_DATA_DIR}/transactions",
        'type': 'csv',
        'target_table': 'recomart.raw.transactions',
        'partition_by': 'transaction_date'
    },
    'products': {
        'path': f"{RAW_DATA_DIR}/products",
        'type': 'json',
        'target_table': 'recomart.raw.products',
        'partition_by': None  # Products don't need date partitioning
    }
}

# Retry configuration
RETRY_CONFIG = {
    'max_retries': 3,
    'retry_delay': 2,  # seconds
    'backoff_multiplier': 2  # exponential backoff
}

# Unity Catalog configuration
CATALOG_NAME = 'recomart'
SCHEMA_NAME = 'raw'

# Create catalog and schema if they don't exist
try:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
    logger.info(f"Catalog '{CATALOG_NAME}' is ready")
    
    spark.sql(f"USE CATALOG {CATALOG_NAME}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_NAME}")
    logger.info(f"Schema '{CATALOG_NAME}.{SCHEMA_NAME}' is ready")
    
    print(f"✅ Unity Catalog configured: {CATALOG_NAME}.{SCHEMA_NAME}")
except Exception as e:
    logger.error(f"Error setting up Unity Catalog: {str(e)}")
    raise

print("\n📋 Data Source Configuration:")
for source, config in DATA_SOURCES.items():
    print(f"  • {source}: {config['type'].upper()} → {config['target_table']}")

In [0]:
def retry_on_failure(max_retries: int = 3, retry_delay: int = 2, backoff_multiplier: int = 2):
    """
    Decorator that implements retry logic with exponential backoff.
    
    Args:
        max_retries: Maximum number of retry attempts
        retry_delay: Initial delay between retries (seconds)
        backoff_multiplier: Multiplier for exponential backoff
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            attempt = 0
            current_delay = retry_delay
            
            while attempt < max_retries:
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    attempt += 1
                    if attempt >= max_retries:
                        logger.error(f"Function {func.__name__} failed after {max_retries} attempts: {str(e)}")
                        raise
                    
                    logger.warning(f"Attempt {attempt}/{max_retries} failed for {func.__name__}: {str(e)}")
                    logger.info(f"Retrying in {current_delay} seconds...")
                    time.sleep(current_delay)
                    current_delay *= backoff_multiplier
            
        return wrapper
    return decorator


@retry_on_failure(**RETRY_CONFIG)
def ingest_csv_file(file_path: str, source_name: str) -> pd.DataFrame:
    """
    Ingest CSV file with error handling and logging.
    
    Args:
        file_path: Path to CSV file
        source_name: Name of the data source (for logging)
    
    Returns:
        DataFrame with ingested data
    """
    logger.info(f"Starting ingestion of {source_name} from {file_path}")
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    
    # Read CSV file
    df = pd.read_csv(file_path)
    
    # Add ingestion metadata
    df['ingestion_timestamp'] = datetime.now()
    df['source_file'] = os.path.basename(file_path)
    df['ingestion_batch_id'] = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    logger.info(f"Successfully ingested {len(df)} records from {source_name}")
    
    return df


@retry_on_failure(**RETRY_CONFIG)
def ingest_json_file(file_path: str, source_name: str) -> pd.DataFrame:
    """
    Ingest JSON file with error handling and logging.
    
    Args:
        file_path: Path to JSON file
        source_name: Name of the data source (for logging)
    
    Returns:
        DataFrame with ingested data
    """
    logger.info(f"Starting ingestion of {source_name} from {file_path}")
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    
    # Read JSON file
    with open(file_path, 'r') as f:
        data = json.load(f)
    
    # Handle different JSON structures
    if isinstance(data, list):
        df = pd.DataFrame(data)
    elif isinstance(data, dict):
        # Check if there's a 'products' or 'data' key
        if 'products' in data:
            df = pd.DataFrame(data['products'])
        elif 'data' in data:
            df = pd.DataFrame(data['data'])
        else:
            # Treat as single record
            df = pd.DataFrame([data])
    else:
        raise ValueError(f"Unsupported JSON structure in {file_path}")
    
    # Add ingestion metadata
    df['ingestion_timestamp'] = datetime.now()
    df['source_file'] = os.path.basename(file_path)
    df['ingestion_batch_id'] = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    logger.info(f"Successfully ingested {len(df)} records from {source_name}")
    
    return df


def save_to_delta(df: pd.DataFrame, table_name: str, partition_by: Optional[str] = None, mode: str = 'append'):
    """
    Save DataFrame to Delta Lake table with partitioning.
    
    Args:
        df: Pandas DataFrame to save
        table_name: Fully qualified table name (catalog.schema.table)
        partition_by: Column to partition by (optional)
        mode: Write mode ('append' or 'overwrite')
    """
    logger.info(f"Saving {len(df)} records to Delta table {table_name}")
    
    try:
        # Convert pandas DataFrame to Spark DataFrame
        spark_df = spark.createDataFrame(df)
        
        # Write to Delta table
        writer = spark_df.write.format('delta').mode(mode)
        
        if partition_by:
            writer = writer.partitionBy(partition_by)
        
        writer.saveAsTable(table_name)
        
        logger.info(f"Successfully saved data to {table_name}")
        print(f"✅ Saved {len(df)} records to {table_name}")
        
    except Exception as e:
        logger.error(f"Error saving to Delta table {table_name}: {str(e)}")
        logger.error(traceback.format_exc())
        raise

print("✅ Helper functions defined:")
print("  • retry_on_failure: Retry decorator with exponential backoff")
print("  • ingest_csv_file: CSV ingestion with error handling")
print("  • ingest_json_file: JSON ingestion with error handling")
print("  • save_to_delta: Delta Lake writer with partitioning")

In [0]:
# Initialize ingestion tracking
ingestion_results = {}

print("\n" + "="*80)
print("📥 INGESTING USER INTERACTIONS")
print("="*80)

try:
    # Get configuration
    config = DATA_SOURCES['user_interactions']
    source_path = config['path']
    
    # Find CSV files in the directory
    csv_files = []
    if os.path.exists(source_path):
        csv_files = [f for f in os.listdir(source_path) if f.endswith('.csv')]
    
    if not csv_files:
        logger.warning(f"No CSV files found in {source_path}")
        print(f"⚠️  No CSV files found in {source_path}")
        print(f"📝 Note: Please ensure user interactions data is available at this location")
        ingestion_results['user_interactions'] = {
            'status': 'skipped',
            'records': 0,
            'error': 'No files found'
        }
    else:
        all_interactions = []
        
        for csv_file in csv_files:
            file_path = os.path.join(source_path, csv_file)
            print(f"\n📄 Processing: {csv_file}")
            
            try:
                # Ingest CSV file
                df = ingest_csv_file(file_path, 'user_interactions')
                
                # Parse date column if exists (for partitioning)
                if 'timestamp' in df.columns:
                    df['interaction_date'] = pd.to_datetime(df['timestamp']).dt.date
                elif 'date' in df.columns:
                    df['interaction_date'] = pd.to_datetime(df['date']).dt.date
                else:
                    # Use ingestion date as partition
                    df['interaction_date'] = datetime.now().date()
                
                all_interactions.append(df)
                print(f"  ✓ Loaded {len(df)} records from {csv_file}")
                
            except Exception as e:
                logger.error(f"Failed to ingest {csv_file}: {str(e)}")
                print(f"  ✗ Failed to load {csv_file}: {str(e)}")
        
        if all_interactions:
            # Combine all DataFrames
            combined_df = pd.concat(all_interactions, ignore_index=True)
            
            # Save to Delta Lake
            save_to_delta(
                combined_df,
                config['target_table'],
                partition_by=config['partition_by'],
                mode='append'
            )
            
            ingestion_results['user_interactions'] = {
                'status': 'success',
                'records': len(combined_df),
                'files_processed': len(all_interactions),
                'table': config['target_table']
            }
            
            print(f"\n✅ User Interactions Ingestion Complete")
            print(f"   Total records: {len(combined_df):,}")
            print(f"   Files processed: {len(all_interactions)}")
        else:
            ingestion_results['user_interactions'] = {
                'status': 'failed',
                'records': 0,
                'error': 'All files failed to load'
            }
            
except Exception as e:
    logger.error(f"Critical error during user interactions ingestion: {str(e)}")
    logger.error(traceback.format_exc())
    ingestion_results['user_interactions'] = {
        'status': 'failed',
        'records': 0,
        'error': str(e)
    }
    print(f"\n❌ User Interactions Ingestion Failed: {str(e)}")

In [0]:
print("\n" + "="*80)
print("📥 INGESTING TRANSACTIONS")
print("="*80)

try:
    # Get configuration
    config = DATA_SOURCES['transactions']
    source_path = config['path']
    
    # Find CSV files in the directory
    csv_files = []
    if os.path.exists(source_path):
        csv_files = [f for f in os.listdir(source_path) if f.endswith('.csv')]
    
    if not csv_files:
        logger.warning(f"No CSV files found in {source_path}")
        print(f"⚠️  No CSV files found in {source_path}")
        print(f"📝 Note: Please ensure transaction data is available at this location")
        ingestion_results['transactions'] = {
            'status': 'skipped',
            'records': 0,
            'error': 'No files found'
        }
    else:
        all_transactions = []
        
        for csv_file in csv_files:
            file_path = os.path.join(source_path, csv_file)
            print(f"\n📄 Processing: {csv_file}")
            
            try:
                # Ingest CSV file
                df = ingest_csv_file(file_path, 'transactions')
                
                # Parse date column if exists (for partitioning)
                if 'timestamp' in df.columns:
                    df['transaction_date'] = pd.to_datetime(df['timestamp']).dt.date
                elif 'date' in df.columns:
                    df['transaction_date'] = pd.to_datetime(df['date']).dt.date
                elif 'transaction_date' not in df.columns:
                    # Use ingestion date as partition
                    df['transaction_date'] = datetime.now().date()
                
                all_transactions.append(df)
                print(f"  ✓ Loaded {len(df)} records from {csv_file}")
                
            except Exception as e:
                logger.error(f"Failed to ingest {csv_file}: {str(e)}")
                print(f"  ✗ Failed to load {csv_file}: {str(e)}")
        
        if all_transactions:
            # Combine all DataFrames
            combined_df = pd.concat(all_transactions, ignore_index=True)
            
            # Save to Delta Lake
            save_to_delta(
                combined_df,
                config['target_table'],
                partition_by=config['partition_by'],
                mode='append'
            )
            
            ingestion_results['transactions'] = {
                'status': 'success',
                'records': len(combined_df),
                'files_processed': len(all_transactions),
                'table': config['target_table']
            }
            
            print(f"\n✅ Transactions Ingestion Complete")
            print(f"   Total records: {len(combined_df):,}")
            print(f"   Files processed: {len(all_transactions)}")
        else:
            ingestion_results['transactions'] = {
                'status': 'failed',
                'records': 0,
                'error': 'All files failed to load'
            }
            
except Exception as e:
    logger.error(f"Critical error during transactions ingestion: {str(e)}")
    logger.error(traceback.format_exc())
    ingestion_results['transactions'] = {
        'status': 'failed',
        'records': 0,
        'error': str(e)
    }
    print(f"\n❌ Transactions Ingestion Failed: {str(e)}")

In [0]:
print("\n" + "="*80)
print("📥 INGESTING PRODUCTS")
print("="*80)

try:
    # Get configuration
    config = DATA_SOURCES['products']
    source_path = config['path']
    
    # Find JSON files in the directory
    json_files = []
    if os.path.exists(source_path):
        json_files = [f for f in os.listdir(source_path) if f.endswith('.json')]
    
    if not json_files:
        logger.warning(f"No JSON files found in {source_path}")
        print(f"⚠️  No JSON files found in {source_path}")
        print(f"📝 Note: Please ensure product data is available at this location")
        ingestion_results['products'] = {
            'status': 'skipped',
            'records': 0,
            'error': 'No files found'
        }
    else:
        all_products = []
        
        for json_file in json_files:
            file_path = os.path.join(source_path, json_file)
            print(f"\n📄 Processing: {json_file}")
            
            try:
                # Ingest JSON file
                df = ingest_json_file(file_path, 'products')
                
                all_products.append(df)
                print(f"  ✓ Loaded {len(df)} records from {json_file}")
                
            except Exception as e:
                logger.error(f"Failed to ingest {json_file}: {str(e)}")
                print(f"  ✗ Failed to load {json_file}: {str(e)}")
        
        if all_products:
            # Combine all DataFrames
            combined_df = pd.concat(all_products, ignore_index=True)
            
            # Remove duplicates based on product_id if it exists
            if 'product_id' in combined_df.columns:
                before_count = len(combined_df)
                combined_df = combined_df.drop_duplicates(subset=['product_id'], keep='last')
                after_count = len(combined_df)
                if before_count > after_count:
                    logger.info(f"Removed {before_count - after_count} duplicate products")
                    print(f"  ℹ️  Removed {before_count - after_count} duplicate products")
            
            # Save to Delta Lake
            save_to_delta(
                combined_df,
                config['target_table'],
                partition_by=config['partition_by'],
                mode='overwrite'  # Overwrite for products as it's master data
            )
            
            ingestion_results['products'] = {
                'status': 'success',
                'records': len(combined_df),
                'files_processed': len(all_products),
                'table': config['target_table']
            }
            
            print(f"\n✅ Products Ingestion Complete")
            print(f"   Total records: {len(combined_df):,}")
            print(f"   Files processed: {len(all_products)}")
        else:
            ingestion_results['products'] = {
                'status': 'failed',
                'records': 0,
                'error': 'All files failed to load'
            }
            
except Exception as e:
    logger.error(f"Critical error during products ingestion: {str(e)}")
    logger.error(traceback.format_exc())
    ingestion_results['products'] = {
        'status': 'failed',
        'records': 0,
        'error': str(e)
    }
    print(f"\n❌ Products Ingestion Failed: {str(e)}")

In [0]:
print("\n" + "="*80)
print("📊 INGESTION SUMMARY REPORT")
print("="*80)
print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n")

# Calculate summary statistics
total_records = sum(r.get('records', 0) for r in ingestion_results.values())
successful_sources = sum(1 for r in ingestion_results.values() if r.get('status') == 'success')
total_sources = len(ingestion_results)

print(f"Overall Statistics:")
print(f"  • Total sources processed: {total_sources}")
print(f"  • Successful ingestions: {successful_sources}")
print(f"  • Failed ingestions: {total_sources - successful_sources}")
print(f"  • Total records ingested: {total_records:,}")
print("\n")

# Detailed results by source
print("Detailed Results by Source:")
print("-" * 80)

for source_name, result in ingestion_results.items():
    status = result.get('status', 'unknown')
    records = result.get('records', 0)
    
    status_icon = "✅" if status == 'success' else "⚠️" if status == 'skipped' else "❌"
    
    print(f"\n{status_icon} {source_name.upper().replace('_', ' ')}")
    print(f"   Status: {status.upper()}")
    print(f"   Records: {records:,}")
    
    if 'files_processed' in result:
        print(f"   Files processed: {result['files_processed']}")
    
    if 'table' in result:
        print(f"   Target table: {result['table']}")
    
    if 'error' in result:
        print(f"   Error: {result['error']}")

print("\n" + "="*80)

# Display sample data from each successfully ingested source
print("\n📋 Sample Data Preview:")
print("="*80)

for source_name, result in ingestion_results.items():
    if result.get('status') == 'success' and result.get('table'):
        try:
            table_name = result['table']
            print(f"\n📦 {source_name.upper().replace('_', ' ')} (from {table_name})")
            
            # Query sample data
            sample_df = spark.sql(f"SELECT * FROM {table_name} LIMIT 5")
            display(sample_df)
            
            # Show table statistics
            count_df = spark.sql(f"SELECT COUNT(*) as total_records FROM {table_name}")
            total = count_df.collect()[0]['total_records']
            print(f"Total records in table: {total:,}")
            
        except Exception as e:
            logger.error(f"Error displaying sample for {source_name}: {str(e)}")
            print(f"⚠️  Could not display sample data: {str(e)}")

print("\n" + "="*80)

# Log completion
logger.info("="*80)
logger.info("Data Ingestion Pipeline - Session Completed")
logger.info(f"Total records ingested: {total_records:,}")
logger.info(f"Successful sources: {successful_sources}/{total_sources}")
logger.info("="*80)

print(f"\n✅ Ingestion pipeline completed!")
print(f"📝 Full logs available at: {log_file}")